# VoxaVersa: train your own AI voice

This notebook turns recordings of **one singer** into a VoxaVersa voice pack (`.vxpack`) that you import
into VoxaVersa and use with the AI voice engine. It runs on Google's free GPUs; you don't need to install
anything on your computer.

**What you need**

* A Google account (the free Colab plan works).
* **10 to 30+ minutes of dry singing by one person**: lead vocals, ad-libs, sustained notes, high and low
  notes. *Dry* means no reverb, no delay, no doubling, no autotune, no backing track. WAV is best; MP3 works.
  Put the files in a `.zip`, or in a folder on your Google Drive.
* The **right to use the voice**: your own voice, a singer who agreed in writing, or sample packs whose
  licence explicitly allows AI voice training.

**How to use it**

1. Menu **Runtime > Change runtime type > T4 GPU > Save**.
2. Fill in the settings in step 2, then run the cells from top to bottom with the play button
   (or **Runtime > Run all**). Each cell says what it is doing and what to check.
3. Training takes about **1.5 to 3 hours** for 300 epochs of 15 to 20 minutes of singing on the free T4. If Colab
   disconnects, open the notebook again and run all cells: with *Google Drive* switched on, training
   continues where it stopped.

Guide with explanations and troubleshooting: `docs/TRAINING.md` in the VoxaVersa repository.

The trainer is the official [RVC WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI)
(MIT licence) at a pinned version, with its pretrained v2 base models and ContentVec/HuBERT
(`lj1995/VoiceConversionWebUI`, MIT).

In [ ]:
#@title 1. Check the GPU { display-mode: "form" }
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
if gpu:
    print("OK - GPU found:", gpu)
else:
    print("PROBLEM: no GPU. Menu Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")
    print("(Training without a GPU would take days.)")

In [ ]:
#@title 2. Settings { display-mode: "form" }
#@markdown **About the voice** (shown in VoxaVersa's voice library)
VOICE_NAME = "My Voice"  #@param {type:"string"}
GENDER = "female"  #@param ["female", "male", "neutral"]
GENRE = "Pop"  #@param {type:"string"}
STYLES = "Smooth, Bright"  #@param {type:"string"}
DESCRIPTION = "AI voice trained on my own recordings"  #@param {type:"string"}
AUTHOR = ""  #@param {type:"string"}
LICENSE = "Personal use only"  #@param {type:"string"}
#@markdown **Your recordings**
DATASET_SOURCE = "upload"  #@param ["upload", "google drive"]
#@markdown If you chose "google drive": a .zip or a folder inside My Drive, e.g. `VoxaVersa/my_vocals.zip`
DRIVE_DATASET_PATH = "VoxaVersa/my_vocals.zip"  #@param {type:"string"}
#@markdown Clean the recordings first (recommended: removes silence, noise, reverb tails, stacked harmonies)
CLEAN_RECORDINGS = True  #@param {type:"boolean"}
#@markdown **Training**
EPOCHS = 300  #@param {type:"integer"}
SAMPLE_RATE = "40k"  #@param ["40k", "48k"]
BATCH_SIZE = 8  #@param {type:"integer"}
SAVE_EVERY_EPOCHS = 10  #@param {type:"integer"}
#@markdown Keep checkpoints on Google Drive so training survives a disconnect (strongly recommended)
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
#@markdown Put the content encoder (378 MB) inside the voice pack. Leave off if you install it once (see the last step).
INCLUDE_ENCODER_IN_PACK = False  #@param {type:"boolean"}
#@markdown **Required:** I made these recordings myself or have explicit permission (and licence) to train an AI voice from them.
I_HAVE_THE_RIGHTS = False  #@param {type:"boolean"}

import re
NAME = re.sub(r"[^a-z0-9]+", "_", VOICE_NAME.lower()).strip("_") or "my_voice"
if not I_HAVE_THE_RIGHTS:
    raise SystemExit("Please confirm I_HAVE_THE_RIGHTS: only train voices you own or have explicit permission to use.")
if EPOCHS < 20:
    print("Note: fewer than 20 epochs rarely gives a usable voice.")
print("Voice '%s' -> files will be called %s.*" % (VOICE_NAME, NAME))

WORK = "/content/work"
if USE_GOOGLE_DRIVE or DATASET_SOURCE == "google drive":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/VoxaVersa"
import os
os.makedirs(WORK, exist_ok=True)
print("Results and checkpoints go to", WORK)

In [ ]:
#@title 3. Install the training software (about 3-5 minutes) { display-mode: "form" }
#@markdown Installs a separate Python 3.12 with PyTorch 2.7.1 (CUDA 11.8) and the pinned packages, so Colab
#@markdown updates can't break it. VoxaVersa's tools come from GitHub.
VOXAVERSA_REPO = "https://github.com/DK7602/VoxaVersa"  #@param {type:"string"}
VOXAVERSA_BRANCH = ""  #@param {type:"string"}
import os, subprocess, sys

def sh(cmd, quiet=False):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=quiet)
    if p.returncode != 0:
        if quiet:
            print(p.stdout[-3000:], p.stderr[-3000:])
        raise SystemExit("Step failed: " + cmd)

if not os.path.isdir("/content/VoxaVersa/tools/training"):
    branch = ("--branch " + VOXAVERSA_BRANCH) if VOXAVERSA_BRANCH else ""
    sh("rm -rf /content/VoxaVersa && git clone -q --depth 1 %s %s /content/VoxaVersa" % (branch, VOXAVERSA_REPO))
if not os.path.isfile("/content/VoxaVersa/tools/training/rvc_train.py"):
    raise SystemExit("This VoxaVersa version has no tools/training - set VOXAVERSA_BRANCH to a branch that has it.")

PY = "/content/rvc-env/bin/python"
if not os.path.exists(PY):
    sh("pip install -q uv", quiet=True)
    sh("uv venv -q --python 3.12 /content/rvc-env", quiet=True)
    sh("uv pip install -q --python %s torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118" % PY, quiet=True)
    sh("uv pip install -q --python %s numpy==1.26.4 scipy==1.17.1 librosa==0.10.2.post1 numba==0.68.0 "
       "soundfile==0.14.0 av==15.1.0 ffmpeg-python==0.2.0 praat-parselmouth==0.4.7 transformers==4.49.0 "
       "tensorboard==2.21.0 matplotlib==3.11.2 scikit-learn==1.9.1 tqdm onnx==1.17.0 onnxruntime==1.20.1" % PY, quiet=True)
check = subprocess.run([PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True).stdout.strip()
print("PyTorch", check)
if not check.endswith("True"):
    print("PROBLEM: PyTorch can't see the GPU - check step 1.")
else:
    print("OK - software installed")

In [ ]:
#@title 4. Download the RVC trainer and its base models (about 1 minute) { display-mode: "form" }
TOOLS = "/content/VoxaVersa/tools/training"
RVC = "/content/rvc"
sh("%s %s/rvc_train.py setup --rvc-dir %s --sr %s" % (PY, TOOLS, RVC, SAMPLE_RATE))
print("OK - trainer ready")

In [ ]:
#@title 5. Your recordings: upload and clean { display-mode: "form" }
#@markdown Upload a .zip (or several audio files) when the button appears. The cleaner keeps only clean,
#@markdown dry, single-voice singing and tells you how many minutes are usable.
import glob, json, os, shutil, zipfile
RAW = "/content/raw"
DATA = os.path.join(WORK, "datasets")
os.makedirs(RAW, exist_ok=True)

already = os.path.isdir(os.path.join(DATA, NAME)) and os.path.exists(os.path.join(DATA, "report.json"))
if DATASET_SOURCE == "upload":
    if not os.listdir(RAW) and not already:
        from google.colab import files
        for name, data in files.upload().items():
            with open(os.path.join(RAW, name), "wb") as f:
                f.write(data)
elif not already:
    src = os.path.join("/content/drive/MyDrive", DRIVE_DATASET_PATH)
    if not os.path.exists(src):
        raise SystemExit("Not found on your Drive: " + src)
    if os.path.isdir(src):
        shutil.copytree(src, RAW, dirs_exist_ok=True)
    else:
        shutil.copy(src, RAW)
for z in glob.glob(os.path.join(RAW, "*.zip")):
    with zipfile.ZipFile(z) as f:
        f.extractall(os.path.join(RAW, os.path.splitext(os.path.basename(z))[0]))
    os.remove(z)

DATASET = os.path.join(DATA, NAME)
REDO = False  #@param {type:"boolean"}
#@markdown (tick REDO to clean the recordings again although a cleaned set from an earlier run exists)
prepared = os.path.isdir(DATASET) and glob.glob(os.path.join(DATASET, "*.wav")) and os.path.exists(os.path.join(DATA, "report.json"))
if prepared and not REDO:
    report = json.load(open(os.path.join(DATA, "report.json")))
    info = report["voices"].get(NAME, {})
    minutes, VOICE_RANGE = info.get("minutes"), info.get("range_midi")
    print("Using the recordings cleaned in an earlier run: %s minutes (%s)" % (minutes, DATASET))
elif CLEAN_RECORDINGS:
    sh("%s %s/prepare_dataset.py %s --out %s --name %s --sr %d --clean" % (
        PY, TOOLS, RAW, DATA, NAME, 48000 if SAMPLE_RATE == "48k" else 40000))
    report = json.load(open(os.path.join(DATA, "report.json")))
    info = report["voices"][NAME]
    minutes = info["minutes"]
    print("\nUsable singing: %.1f minutes in %d clips, range %s" % (minutes, info["clips"], info.get("range_notes", "?")))
    if report["rejected_seconds"]:
        print("Removed:", ", ".join("%s %.0f s" % kv for kv in report["rejected_seconds"].items()),
              "(details: %s/report.md)" % DATA)
    VOICE_RANGE = info.get("range_midi")
else:
    shutil.copytree(RAW, DATASET, dirs_exist_ok=True)
    minutes, VOICE_RANGE = None, None
if minutes is not None and minutes < 3:
    raise SystemExit("Less than 3 minutes of usable singing - that is not enough for a voice. Add more recordings.")
if minutes is not None and minutes < 10:
    print("Note: under 10 minutes works, but the voice will be less stable. 20-30+ minutes is much better.")

In [ ]:
#@title 6. Train (1.5 - 3 hours; you can close the browser tab while it runs, but keep Colab connected) { display-mode: "form" }
#@markdown Progress is printed every epoch. If Colab disconnects: run all cells again - training resumes
#@markdown from the last checkpoint on your Drive. Training stops by itself after EPOCHS epochs.
import os, re, subprocess, time, glob
exp_store = os.path.join(WORK, "training", NAME)
os.makedirs(exp_store, exist_ok=True)
exp = os.path.join(RVC, "logs", NAME)
if not os.path.islink(exp):
    if os.path.isdir(exp):
        shutil.rmtree(exp)
    os.symlink(exp_store, exp)
weights_store = os.path.join(WORK, "training", NAME + "_weights")
os.makedirs(weights_store, exist_ok=True)

cmd = [PY, TOOLS + "/rvc_train.py", "train", "--rvc-dir", RVC, "--dataset", DATASET, "--name", NAME,
       "--sr", SAMPLE_RATE, "--epochs", str(EPOCHS), "--batch", str(BATCH_SIZE),
       "--save-every", str(SAVE_EVERY_EPOCHS)]
if minutes is None or minutes < 45:
    cmd.append("--cache-gpu")      # keeps the whole dataset in GPU memory: faster on Colab's 2 CPU cores
start = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
noise = re.compile(r"Warning|warn\(|torch.load|weight_norm|^\s*$|socket|leaked semaphore|DEBUG")
for line in proc.stdout:
    if noise.search(line):
        continue
    m = re.search(r"====> .*?(\d+)", line)
    if m or "loss_mel" in line or "[rvc_train]" in line or "Error" in line or "error" in line:
        print(line.rstrip()[:200], flush=True)
    if m:
        epoch = int(m.group(1))
        mins = (time.time() - start) / 60
        print("    epoch %d of %d - %.0f min so far" % (epoch, EPOCHS, mins), flush=True)
        # keep only the 3 newest intermediate models (55 MB each), copied to Drive
        small = sorted(glob.glob(os.path.join(RVC, "assets/weights", NAME + "_e*_s*.pth")), key=os.path.getmtime)
        for f in small[-1:]:
            shutil.copy(f, weights_store)
        for f in sorted(glob.glob(os.path.join(weights_store, NAME + "_e*_s*.pth")), key=os.path.getmtime)[:-3]:
            os.remove(f)
proc.wait()
if proc.returncode != 0:
    raise SystemExit("Training stopped with an error (see above). Run this cell again to continue.")
final = os.path.join(RVC, "assets/weights", NAME + ".pth")
if os.path.exists(final):
    shutil.copy(final, weights_store)
print("OK - training finished")

In [ ]:
#@title 7. Export for VoxaVersa (ONNX) and listen { display-mode: "form" }
#@markdown Converts the trained model and the content encoder to ONNX (and checks them against PyTorch),
#@markdown then converts one of your clips with it so you can hear the result.
#@markdown Set USE_LATEST_CHECKPOINT to export a model while training is still running / was interrupted.
USE_LATEST_CHECKPOINT = False  #@param {type:"boolean"}
TRANSPOSE_TEST = 0  #@param {type:"integer"}
OUT = os.path.join(WORK, "export")
os.makedirs(OUT, exist_ok=True)
onnx_path = os.path.join(OUT, NAME + ".onnx")
pth = os.path.join(RVC, "assets/weights", NAME + ".pth")
if USE_LATEST_CHECKPOINT or not os.path.exists(pth):
    pth = os.path.join(RVC, "logs", NAME, "G_2333333.pth")
    print("Using the latest training checkpoint")
sh("%s %s/export_onnx.py --rvc-dir %s --pth %s --out %s" % (PY, TOOLS, RVC, pth, onnx_path))
ENCODER = os.path.join(WORK, "contentvec-768.onnx")
if not os.path.exists(ENCODER):
    sh("%s %s/export_encoder.py --model %s/assets/hubert_base --out %s" % (PY, TOOLS, RVC, ENCODER))

clip = sorted(glob.glob(os.path.join(DATASET, "*.wav")))[0]
preview = os.path.join(OUT, NAME + "_preview.wav")
sh("%s %s/try_voice.py --model %s --encoder %s --input %s --out %s --rvc-dir %s --transpose %d" % (
    PY, TOOLS, onnx_path, ENCODER, clip, preview, RVC, TRANSPOSE_TEST))
from IPython.display import Audio, display
print("Original:"); display(Audio(clip))
print("Your AI voice (a clip it was trained on - try another singer's dry vocal with try_voice.py too):"); display(Audio(preview))

In [ ]:
#@title 8. Make the voice pack and download it { display-mode: "form" }
pack = os.path.join(WORK, NAME + ".vxpack")
args = [PY, "/content/VoxaVersa/tools/neural/package_voice.py", "--model", onnx_path, "--name", VOICE_NAME,
        "--out", pack, "--gender", GENDER, "--genre", GENRE, "--styles", STYLES, "--description", DESCRIPTION,
        "--author", AUTHOR, "--license", LICENSE, "--sample-rate", "48000" if SAMPLE_RATE == "48k" else "40000",
        "--encoder", "contentvec-768", "--i-have-the-rights"]
if VOICE_RANGE:
    args += ["--range", str(VOICE_RANGE[0]), str(VOICE_RANGE[1])]
if INCLUDE_ENCODER_IN_PACK:
    args += ["--encoder-file", ENCODER]
print(subprocess.run(args, capture_output=True, text=True).stdout)
from google.colab import files
files.download(pack)
if not INCLUDE_ENCODER_IN_PACK:
    print("Also download the content encoder once (all voices share it):", ENCODER)
    files.download(ENCODER)
print("Saved on your Drive too:", pack)

## Install the voice in VoxaVersa

1. **ONNX Runtime** (once): download `onnxruntime-<your system>-1.20.1` from
   <https://github.com/microsoft/onnxruntime/releases/tag/v1.20.1> and copy the library into VoxaVersa's
   `Runtime` folder (Windows `%APPDATA%\VoxaVersa\Runtime\onnxruntime.dll`, macOS
   `~/Library/Application Support/VoxaVersa/Runtime/libonnxruntime.1.20.1.dylib`, Linux
   `~/.config/VoxaVersa/Runtime/libonnxruntime.so.1`).
2. **Content encoder** (once, unless it is inside the pack): copy `contentvec-768.onnx` into the `Encoders`
   folder next to it (`%APPDATA%\VoxaVersa\Encoders\`, `~/Library/Application Support/VoxaVersa/Encoders/`,
   `~/.config/VoxaVersa/Encoders/`).
3. In VoxaVersa: settings menu > **Import voice pack** > choose the `.vxpack`. Select the voice and switch
   the **AI engine** on.

Problems? See *Troubleshooting* in `docs/TRAINING.md`.